# TRELLIS.2 – Text/Image to 3D (بدون Flash Attention)
تشغيل مشروع [TRELLIS.2-Text-to-3D-FA2](https://github.com/PRITHIVSAKTHIUR/TRELLIS.2-Text-to-3D-FA2) على Google Colab.

✅ **يعمل على أي GPU (T4, A100, V100...)** دون الحاجة إلى Flash Attention.
يستخدم الانتباه الافتراضي `SDPA` من PyTorch مما يوفر توافقاً كاملاً وسرعة جيدة.

In [ ]:
import torch
import subprocess

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if not torch.cuda.is_available():
    raise RuntimeError("الرجاء تفعيل GPU من Runtime → Change runtime type → GPU")

!nvidia-smi
gpu_name = subprocess.getoutput("nvidia-smi --query-gpu=name --format=csv,noheader")
print(f"GPU: {gpu_name}")
print("✅ سيتم استخدام SDPA (الانتباه الافتراضي) – لا حاجة لـ Flash Attention.")

In [ ]:
import os, sys

# استنساخ المستودع إذا لم يكن موجوداً
if not os.path.exists("TRELLIS.2-Text-to-3D-FA2"):
    !git clone https://github.com/PRITHIVSAKTHIUR/TRELLIS.2-Text-to-3D-FA2.git
%cd TRELLIS.2-Text-to-3D-FA2

# تثبيت spconv (ضروري جداً)
cuda_ver = torch.version.cuda.replace(".", "")
try:
    !pip install spconv-cu{cuda_ver}
except:
    print("محاولة تثبيت spconv العام...")
    !pip install spconv

# تثبيت المتطلبات الأساسية
!pip install -r requirements.txt 2>/dev/null || echo "لا يوجد requirements.txt"
!pip install trimesh pygltflib viser omegaconf opencv-python imageio[ffmpeg] gradio huggingface_hub 2>&1 | tail -5

# لا نقوم بتثبيت flash-attn على الإطلاق
print("✅ تم تثبيت جميع المتطلبات (بدون Flash Attention).")

In [ ]:
import sys, os

# إضافة المستودع إلى مسار Python
repo_path = "/content/TRELLIS.2-Text-to-3D-FA2"
if repo_path not in sys.path:
    sys.path.insert(0, repo_path)

import torch
from trellis.pipelines import TrellisImageTo3DPipeline

# تحميل النموذج مع إجباره على استخدام SDPA بدلاً من flash-attn
pipeline = TrellisImageTo3DPipeline.from_pretrained(
    "JeffreyXiang/TRELLIS-image-large",
    torch_dtype=torch.float16,
    attn_implementation="sdpa"   # ← هنا نجبر استخدام الانتباه الافتراضي
)
pipeline.to("cuda")
print("✅ تم تحميل خط الأنابيب بنجاح (باستخدام SDPA).")

In [ ]:
from PIL import Image
import requests

# تحميل صورة تجريبية (استبدلها بصورتك إذا أردت)
url = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/diffusers/tiger.png"
image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
display(image)

In [ ]:
# توليد المجسم الثلاثي الأبعاد
outputs = pipeline.run(
    image,
    seed=42
)
print("🔹 تم الانتهاء من التوليد")
print("المفاتيح الناتجة:", outputs.keys())
gaussian = outputs.get('gaussian')
mesh = outputs.get('mesh')

In [ ]:
from pathlib import Path

out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

# تصدير الشبكة (mesh) بصيغة GLB
if mesh is not None:
    glb_path = out_dir / "model.glb"
    mesh.export(glb_path)
    print(f"✅ تم تصدير الشبكة إلى {glb_path}")
else:
    print("⚠️ لا توجد شبكة في المخرجات")

# تصدير السحابة الغاوسية بصيغة PLY (إذا كانت مدعومة)
if gaussian is not None:
    ply_path = out_dir / "gaussian.ply"
    if hasattr(gaussian[0], 'save_ply'):
        gaussian[0].save_ply(ply_path)
        print(f"✅ تم تصدير Gaussians إلى {ply_path}")
    else:
        print("⚠️ لا توجد دالة save_ply، يمكن استخدام xyz يدوياً لاحقاً.")

In [ ]:
# إعداد العارض التفاعلي باستخدام viser + منفذ Colab الداخلي (بدون ngrok)
import viser
from google.colab.output import eval_js
from IPython.display import IFrame, display

# تشغيل خادم viser على منفذ 8080
server = viser.ViserServer(host="0.0.0.0", port=8080)

# إضافة المحتوى إلى المشهد
if gaussian is not None and len(gaussian) > 0:
    try:
        from trellis.utils import render_utils
        render_utils.render_gaussian(server, gaussian[0])
        print("✅ تم عرض السحابة الغاوسية باستخدام render_utils")
    except ImportError:
        print("⚠️ تعذر استيراد render_utils، جارٍ إضافة سحابة نقاط بسيطة...")
        points = gaussian[0].xyz.detach().cpu().numpy().reshape(-1, 3)
        server.scene.add_point_cloud(
            "/points",
            points=points,
            colors=(255, 200, 200),
            point_size=0.01
        )
        print("✅ تم إضافة سحابة نقاط مبدئية")
else:
    print("لا توجد بيانات غاوسية للعرض")

# إنشاء رابط العرض من Colab
url = eval_js(f"google.colab.kernel.proxyPort(8080)")
public_url = f"https://{url}"
print(f"🔗 رابط المشاهدة التفاعلية: {public_url}")

# عرض العارض داخل الدفتر
display(IFrame(src=public_url, width="100%", height="600px"))
print("إذا لم يظهر المشهد، افتح الرابط أعلاه في نافذة جديدة.")

## ملاحظات
- تم تعطيل Flash Attention بالكامل – النموذج يعمل على **T4 المجاني** بكفاءة.
- إذا كنت تستخدم A100 وأردت تفعيل Flash Attention، أعد تعيين `attn_implementation="flash_attention_2"`.
- الملفات المصدرة ستظهر في مجلد `outputs` ويمكن تحميلها من متصفح الملفات في Colab.